# RockWatch - Phase 3: Baseline Models
**Project:** Intelligent Landslide Susceptibility Prediction System for Himachal Pradesh  
**Objective:** Train and benchmark lower-bound and interpretable baseline models (`DummyClassifier`, `LogisticRegression`, `DecisionTreeClassifier`).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import roc_curve, auc

# Set project root
PROJECT_ROOT = Path('.').resolve().parents[0] if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_processed_data
from src.evaluation.metrics import evaluate_predictions, find_optimal_threshold
from src.models.baseline import (
    create_dummy_baseline,
    create_logistic_regression,
    create_decision_tree,
    ALL_FEATURES,
    LR_FEATURES,
)

## 1. Load Processed Datasets

In [ ]:
f_train = load_processed_data('train', features_only=True, processed_dir=PROJECT_ROOT / 'data' / 'processed')
f_val = load_processed_data('val', features_only=True, processed_dir=PROJECT_ROOT / 'data' / 'processed')
f_test = load_processed_data('test', features_only=True, processed_dir=PROJECT_ROOT / 'data' / 'processed')

y_train = f_train['class'].values
y_val = f_val['class'].values
y_test = f_test['class'].values

print(f'Train samples: {len(f_train):,} | Landslides: {(y_train==1).sum():,}')
print(f'Val samples  : {len(f_val):,} | Landslides: {(y_val==1).sum():,}')
print(f'Test samples : {len(f_test):,} | Landslides: {(y_test==1).sum():,}')

## 2. Load and Inspect Baseline Benchmark Results

In [ ]:
res_df = pd.read_csv(PROJECT_ROOT / 'models' / 'evaluation' / 'baseline_results.csv')
res_df

## 3. Logistic Regression Coefficients & Odds Ratios

In [ ]:
lr_model = joblib.load(PROJECT_ROOT / 'models' / 'trained' / 'logistic_regression_v1.joblib')
coef_df = pd.DataFrame({
    'Feature': LR_FEATURES,
    'Coefficient': lr_model.coef_[0],
    'Odds_Ratio': np.exp(lr_model.coef_[0]),
}).sort_values('Coefficient', ascending=False)

print(f'Intercept: {lr_model.intercept_[0]:.4f}')
coef_df

## 4. Receiver Operating Characteristic (ROC) Comparison

In [ ]:
models = {
    'Dummy (Prior)': (joblib.load(PROJECT_ROOT / 'models' / 'trained' / 'dummy_baseline.joblib'), ALL_FEATURES),
    'Logistic Regression': (lr_model, LR_FEATURES),
    'Decision Tree (d=5)': (joblib.load(PROJECT_ROOT / 'models' / 'trained' / 'decision_tree_v1.joblib'), ALL_FEATURES),
}

fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
for name, (clf, feats) in models.items():
    p_test = clf.predict_proba(f_test[feats].values)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, p_test)
    ax.plot(fpr, tpr, label=f'{name} (Test AUC = {auc(fpr, tpr):.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_title('Baseline Models ROC on Held-out Spatial Test Split (Chamba)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
ax.grid(True, linestyle=':', alpha=0.6)
plt.show()